# HW1 - Data-quality report for `shop.orders` - SOLUTION
**Slides 11, 15, 26.** Try [`../HW1_data_quality_report.ipynb`](../HW1_data_quality_report.ipynb) first, then compare.

**Before you run it:** `bash lab/reset.sh` from the repo folder on your laptop, so the numbers match the self-check.

In [ ]:
import os
from pymongo import MongoClient

# Inside dev_env the host is "mongo" (Jupyter does not see $MONGO_URI, so we fall back to the same value).
URI = os.getenv("MONGO_URI", "mongodb://mongo:27017/?directConnection=true")
client = MongoClient(URI, serverSelectionTimeoutMS=5000)
db = client["shop"]
coll = db["orders"]
print("connected to", URI, "| orders:", coll.count_documents({}))

## 1. Type profiles (slides 11, 26)
Return a dict `{type_name: count}` using the **MongoDB `$type` names** (`"decimal"`, `"double"`, `"string"`, `"missing"`, `"date"`, `"long"`, ...).
Hint: the `$group` on `{"$type": "$total"}` from A2 works the same from Python.

In [ ]:
def type_profile(field):
    rows = coll.aggregate([{"$group": {"_id": {"$type": f"${field}"}, "n": {"$sum": 1}}}])
    return {r["_id"]: r["n"] for r in rows}

## 2. Missing status, duplicate keys, email shapes, empty items (slides 15, 26)

In [ ]:
def missing_status():
    return coll.count_documents({"status": {"$exists": False}})

def duplicate_order_ids():
    rows = coll.aggregate([
        {"$group": {"_id": "$order_id", "n": {"$sum": 1}}},
        {"$match": {"n": {"$gt": 1}}},
    ])
    return sorted(r["_id"] for r in rows)

def email_shapes():
    return {
        "customer.email": coll.count_documents({"customer.email": {"$exists": True}}),
        "customer.contact.email": coll.count_documents({"customer.contact.email": {"$exists": True}}),
    }

def empty_items():
    return coll.count_documents({"items": {"$size": 0}})

## 3. Build and print the report

In [ ]:
report = {
    "total_types": type_profile("total"),
    "created_at_types": type_profile("created_at"),
    "missing_status": missing_status(),
    "duplicate_order_ids": duplicate_order_ids(),
    "email_shapes": email_shapes(),
    "empty_items": empty_items(),
}
for key, value in report.items():
    print(f"{key:20}: {value}")

## Self-check (do not edit)

In [ ]:
EXPECTED = {
    "total_types": {"decimal": 115, "double": 55, "string": 19, "missing": 11},
    "created_at_types": {"date": 162, "string": 26, "long": 12},
    "missing_status": 7,
    "duplicate_order_ids": ["ORD-10010", "ORD-10011"],
    "email_shapes": {"customer.email": 107, "customer.contact.email": 93},
    "empty_items": 40,
}
for key, expected in EXPECTED.items():
    assert report[key] == expected, f"{key}: got {report[key]!r}, expected {expected!r} (did you run lab/reset.sh?)"
    print("OK", key)
print("HW1 self-check passed")

**Reflection - model answer:** fail the load on things that break the Silver contract or the MERGE: a new `$type` showing up for `total` or `created_at`, duplicate keys you cannot resolve, or a sudden jump in missing status or email. Only warn on known, tolerated mess that Silver already converts (string totals, three date formats, empty items).